# Lab 1 Part 2 — Tool Calling, ReAct, and SQL Agents

**Day 1 Morning Extension | ~45 minutes | CPU | OpenAI API key required**

Part 1 showed the LLM as an inference engine: you send text, it returns text. Part 2 changes the shape of the system. Now the LLM can ask your program to run a tool. Your program performs the action, sends the result back, and the LLM uses that observation to answer.

This is the foundation underneath practical agents. We will build the idea in three steps:

1. **Tool calling:** the model requests a structured function call instead of guessing.
2. **ReAct:** the model alternates between reasoning and acting.
3. **Real API tool:** the model calls a no-key weather API through your code.
4. **SQL agent:** the model turns a natural-language question into a safe database query, your code runs it, and the model explains the result.

> The big idea: the model does not directly touch your database or APIs. You expose carefully described tools. The model asks to use them. Your code decides what actually runs.


In [ ]:
!pip install -q openai pandas httpx
print('Install complete')


In [ ]:
# Configuration — works in Colab Secrets or local environment variables
import os

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. In Colab, open the key icon in the left sidebar, "
            f"add a secret named {name}, paste the value, and enable Notebook access."
        )
    return value

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
DEFAULT_MODEL = 'gpt-4o-mini'

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

print(f'Config loaded — OPENAI_API_KEY starts with {OPENAI_API_KEY[:8]}...')


## Part A — From Text Generation to Tool Use

A normal chat completion asks the model to answer directly. Tool calling gives the model another option: it can return a structured request such as:

```json
{
  "name": "estimate_model_memory",
  "arguments": {"params_b": 7, "precision": "int4"}
}
```

That request is not executed by the model. It is executed by your Python code. This boundary matters:

- The **model decides** which tool it wants and what arguments to pass.
- Your **application validates and executes** the tool.
- The **model observes** the tool result and writes the final answer.

This is why tool/function descriptions are part of your product surface. If the tool schema is vague, the model will use it badly.


In [ ]:
# A tiny tool: estimate model memory from parameter count and precision.
# This is deliberately simple so we can see the tool-calling mechanics clearly.
import json

def estimate_model_memory(params_b: float, precision: str) -> str:
    bytes_per_param = {
        'fp32': 4.0,
        'fp16': 2.0,
        'bf16': 2.0,
        'int8': 1.0,
        'int4': 0.5,
        'nf4': 0.5,
    }
    key = precision.lower()
    if key not in bytes_per_param:
        return json.dumps({'error': f'Unsupported precision: {precision}'})
    memory_gb = params_b * 1e9 * bytes_per_param[key] / 1e9
    return json.dumps({
        'params_b': params_b,
        'precision': key,
        'estimated_weight_memory_gb': round(memory_gb, 2),
    })

print(estimate_model_memory(7, 'int4'))


### Describe the Tool to the Model

The model cannot inspect your Python function. It only sees the schema you send in the API call. The schema tells it:

- the function name,
- what the function does,
- what arguments it accepts,
- which arguments are required.

This is a contract. The better the contract, the better the tool use.


In [ ]:
memory_tool = {
    'type': 'function',
    'function': {
        'name': 'estimate_model_memory',
        'description': (
            'Estimate the weight memory in GB for an LLM from parameter count and precision. '
            'Use this for deployment sizing questions.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'params_b': {
                    'type': 'number',
                    'description': 'Model size in billions of parameters, such as 7 or 13.'
                },
                'precision': {
                    'type': 'string',
                    'enum': ['fp32', 'fp16', 'bf16', 'int8', 'int4', 'nf4'],
                    'description': 'The numeric precision used to store model weights.'
                },
            },
            'required': ['params_b', 'precision'],
        },
    },
}


### The Two-Round Tool Calling Loop

Before wrapping anything in a helper function, we will inspect the mechanics one cell at a time. This matters because `finish_reason` changes meaning:

- Without tools, the model usually stops with `finish_reason='stop'` and returns text.
- With tools, the model may stop with `finish_reason='tool_calls'` and return no final answer yet.

That second case means: **the model is asking your application to act.**


In [ ]:
# TC-1 — Normal response without tools: the model answers directly.
plain_messages = [
    {'role': 'user', 'content': 'How much memory does a 7B INT4 model need?'}
]
plain = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=plain_messages,
)
print('finish_reason:', plain.choices[0].finish_reason)
print('content:')
print(plain.choices[0].message.content)


In the normal response, the model gives an answer immediately. It may calculate correctly, but it is still generating from its learned patterns. There is no external computation and no explicit tool result.


In [ ]:
# TC-2 — Add a tool: the model can ask your code to run a function.
tool_messages = [
    {'role': 'user', 'content': 'How much memory does a 7B INT4 model need?'}
]
tool_response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=tool_messages,
    tools=[memory_tool],
    tool_choice={'type': 'function', 'function': {'name': 'estimate_model_memory'}},
)

choice = tool_response.choices[0]
print('finish_reason:', choice.finish_reason)
print('content:', choice.message.content)
print('tool_calls:')
print(choice.message.tool_calls)


When `finish_reason` is `tool_calls`, the model has not finished answering. It has paused and returned one or more structured action requests. Your application now owns the next step.

Important API rule: if the assistant message contains multiple `tool_calls`, your message history must include one `role='tool'` response for **every** `tool_call_id` before you ask the model to continue. Comparison questions often trigger parallel tool calls, such as one call for INT4 and another call for FP16.

In [ ]:
# TC-3 — Manual Round 2: run every requested tool and return every observation.
assistant_message = tool_response.choices[0].message
round2_messages = [
    {'role': 'user', 'content': 'How much memory does a 7B INT4 model need?'},
    assistant_message,
]

for tool_call in assistant_message.tool_calls:
    args = json.loads(tool_call.function.arguments)
    print('Model wants to call:', tool_call.function.name)
    print('Arguments:', args)

    if tool_call.function.name != 'estimate_model_memory':
        tool_result = json.dumps({'error': f'Unknown tool: {tool_call.function.name}'})
    else:
        tool_result = estimate_model_memory(**args)

    print('Tool result:', tool_result)
    round2_messages.append({
        'role': 'tool',
        'tool_call_id': tool_call.id,
        'content': tool_result,
    })

final = client.chat.completions.create(model=DEFAULT_MODEL, messages=round2_messages)
print('finish_reason:', final.choices[0].finish_reason)
print('Final answer:')
print(final.choices[0].message.content)

Now the two-round loop should be visible. The helper function below does the same thing, but now you know exactly what it automates.

Notice that this is still a **one-tool agent** in the sense that it exposes one Python function, `estimate_model_memory`. But the model may call that same tool multiple times in one assistant turn. The wrapper must therefore loop over all tool calls before making Round 2.

In [ ]:
# TC-4 — Wrap the mechanics in a reusable one-tool agent.
def memory_agent(question: str) -> str:
    messages = [
        {'role': 'system', 'content': 'You answer LLM deployment sizing questions. Use tools when helpful.'},
        {'role': 'user', 'content': question},
    ]

    first = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        tools=[memory_tool],
        tool_choice='auto',
    )
    first_message = first.choices[0].message

    if not first_message.tool_calls:
        return first_message.content

    messages.append(first_message)

    for tool_call in first_message.tool_calls:
        args = json.loads(tool_call.function.arguments)
        print(f"Tool requested: {tool_call.function.name}({args})")

        if tool_call.function.name != 'estimate_model_memory':
            tool_result = json.dumps({'error': f'Unknown tool: {tool_call.function.name}'})
        else:
            tool_result = estimate_model_memory(**args)

        messages.append({
            'role': 'tool',
            'tool_call_id': tool_call.id,
            'content': tool_result,
        })

    second = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages)
    return second.choices[0].message.content

print(memory_agent('How much weight memory does a 7B model need in INT4 versus FP16?'))

## Part B — ReAct: Reasoning + Acting

ReAct means **Reason + Act**. The model does not only produce an answer. It works in a loop:

```text
Thought: What do I need to know?
Action: Call a tool.
Observation: Read the tool result.
Thought: Do I have enough information?
Answer: Respond to the user.
```

Older ReAct demos often make the model print text like `Action: calculate[2+2]`, then parse that text with regular expressions. That is useful for understanding the idea, and the older bonus notebook shows that style. Modern API tool calling gives us a more reliable version: the **Action** is returned as structured JSON, not as fragile text.

The SQL agent below is ReAct in production clothing: reason about the question, act by querying a database, observe rows, answer.


## Part C — A Real External API Tool: Weather

Tools are not limited to local Python math. They can call web APIs, internal services, filesystems, databases, or deployment platforms. This example uses Open-Meteo, a free weather API that does not require an API key.

The interesting part is that the user can ask for weather in a city, while the tool requires latitude and longitude. The model uses its general knowledge to choose coordinates; the tool provides current external data.


In [ ]:
import httpx

def get_current_weather(latitude: float, longitude: float) -> str:
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'current_weather': True,
    }
    response = httpx.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.text

weather_tool = {
    'type': 'function',
    'function': {
        'name': 'get_current_weather',
        'description': 'Get current weather for a latitude and longitude using the Open-Meteo API.',
        'parameters': {
            'type': 'object',
            'properties': {
                'latitude': {'type': 'number', 'description': 'Latitude of the location.'},
                'longitude': {'type': 'number', 'description': 'Longitude of the location.'},
            },
            'required': ['latitude', 'longitude'],
        },
    },
}


In [ ]:
weather_messages = [
    {'role': 'user', 'content': 'What is the current weather in Amman, Jordan?'}
]
weather_response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=weather_messages,
    tools=[weather_tool],
    tool_choice={'type': 'function', 'function': {'name': 'get_current_weather'}},
)

choice = weather_response.choices[0]
print('finish_reason:', choice.finish_reason)

final_weather_messages = [weather_messages[0], choice.message]
for call in choice.message.tool_calls:
    args = json.loads(call.function.arguments)
    print('Tool requested:', call.function.name)
    print('Arguments:', args)

    if call.function.name != 'get_current_weather':
        weather_result = json.dumps({'error': f'Unknown tool: {call.function.name}'})
    else:
        weather_result = get_current_weather(**args)

    print('Weather API result:', weather_result[:300])
    final_weather_messages.append({
        'role': 'tool',
        'tool_call_id': call.id,
        'content': weather_result,
    })

final_weather = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=final_weather_messages,
)
print('Final answer:')
print(final_weather.choices[0].message.content)

In [ ]:
# display markdown
from IPython.display import display, Markdown
display(Markdown(final_weather.choices[0].message.content))

## Part D — SQL Agent From Scratch

Now we build the practical version. The user asks a natural-language question about model benchmarks. The LLM does not know the table contents. Instead, it gets a `run_sql` tool and a schema description.

The flow is:

```text
User question
  -> model chooses a SELECT query
  -> Python validates and runs the query
  -> SQLite returns rows
  -> model explains the result
```

This is the core of a SQL agent. Frameworks like LangChain add convenience, retries, memory, validation, tracing, and connectors. The core loop is what you are about to build by hand.


In [ ]:
# Build a small in-memory benchmark database.
# SQLite is built into Python, so this works cleanly in Colab.
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE models (name TEXT, params_b REAL, vram_gb REAL, tokens_per_sec INTEGER, quality REAL)')
conn.executemany('INSERT INTO models VALUES (?,?,?,?,?)', [
    ('Qwen2.5-0.5B',  0.5,  2.0, 120, 6.2),
    ('Qwen2.5-1.5B',  1.5,  4.0,  85, 7.1),
    ('Qwen2.5-7B',    7.0, 14.0,  40, 8.3),
    ('Llama-3.2-3B',  3.0,  6.0,  65, 7.4),
    ('Llama-3.1-8B',  8.0, 16.0,  35, 8.5),
    ('Mistral-7B',    7.0, 14.0,  42, 8.1),
])
conn.commit()

pd.read_sql_query('SELECT * FROM models', conn)


### Build a Safe SQL Tool

A real SQL agent must be constrained. Never hand an LLM arbitrary write access to a production database. In this classroom version, we enforce the minimum safety rules:

- only `SELECT` queries,
- one statement at a time,
- no comments or semicolon chaining,
- row limit applied by the application,
- error messages returned as data instead of crashing the notebook.

These guardrails are not optional in production. They are the difference between a demo and a system you can trust.


In [ ]:
import re

def is_safe_select(query: str) -> tuple[bool, str]:
    normalized = ' '.join(query.strip().split()).lower()
    if not normalized.startswith('select '):
        return False, 'Only SELECT queries are allowed.'
    blocked = [';', '--', '/*', '*/', ' pragma ', ' attach ', ' detach ', ' drop ', ' delete ', ' update ', ' insert ', ' alter ', ' create ']
    if any(token in f' {normalized} ' for token in blocked):
        return False, 'Query contains a blocked token or multiple-statement pattern.'

    table_refs = re.findall(r'\b(?:from|join)\s+([a-zA-Z_][\w]*)', normalized)
    if not table_refs:
        return False, 'Query must read from the models table.'
    if any(table != 'models' for table in table_refs):
        return False, 'Query may only read from the models table.'
    return True, 'ok'

def run_sql(query: str) -> str:
    ok, reason = is_safe_select(query)
    if not ok:
        return json.dumps({'error': reason, 'query': query})
    try:
        cur = conn.execute(query)
        rows = cur.fetchmany(20)
        cols = [d[0] for d in cur.description]
        return json.dumps([dict(zip(cols, row)) for row in rows])
    except Exception as e:
        return json.dumps({'error': str(e), 'query': query})

print(run_sql('SELECT name, vram_gb FROM models WHERE vram_gb <= 8 ORDER BY vram_gb'))
print(run_sql('SELECT name FROM models_v2'))


### Describe the Database Tool

The tool description gives the model enough schema context to write useful SQL. Notice that we include table and column names in the description. Without that, the model has to guess.


In [ ]:
sql_tool = {
    'type': 'function',
    'function': {
        'name': 'run_sql',
        'description': (
            'Run a read-only SQLite SELECT query against the models table. '
            'The table columns are: name TEXT, params_b REAL, vram_gb REAL, '
            'tokens_per_sec INTEGER, quality REAL from 0 to 10. '
            'Only use SELECT queries. Do not use semicolons.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'A single read-only SQLite SELECT query over the models table.'
                }
            },
            'required': ['query'],
        },
    },
}


### The SQL Agent Loop

Read this slowly. There is no framework here:

1. We send the user question plus the SQL tool schema.
2. The model requests a SQL query.
3. We validate and execute the query.
4. We send rows back as the tool observation.
5. The model writes the final answer.

That is the heart of a SQL agent.


In [ ]:
def sql_agent(question: str) -> str:
    messages = [
        {
            'role': 'system',
            'content': (
                'You answer questions about LLM benchmark data. '
                'Use the run_sql tool when the answer requires table data. '
                'When you answer, mention the evidence from the SQL result.'
            ),
        },
        {'role': 'user', 'content': question},
    ]

    # Round 1: model decides what query would answer the question.
    first = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        tools=[sql_tool],
        tool_choice='auto',
        temperature=0,
    )
    first_message = first.choices[0].message

    if not first_message.tool_calls:
        return first_message.content

    messages.append(first_message)

    for call in first_message.tool_calls:
        args = json.loads(call.function.arguments)

        if call.function.name != 'run_sql':
            print(f'Unknown tool requested: {call.function.name}')
            result = json.dumps({'error': f'Unknown tool: {call.function.name}'})
        else:
            query = args['query']
            print(f'SQL requested: {query}')
            result = run_sql(query)

        print(f'Tool observation: {result[:180]}...')
        messages.append({
            'role': 'tool',
            'tool_call_id': call.id,
            'content': result,
        })

    # Round 2: model reads the observation and answers in natural language.
    second = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        temperature=0.2,
    )
    return second.choices[0].message.content

In [ ]:
questions = [
    'Which models fit in 8 GB of VRAM?',
    'What is the fastest model with quality above 8.0?',
    'What is the average quality of models under 5B parameters?',
]

for q in questions:
    print(f'Q: {q}')
    print(sql_agent(q))
    print('-' * 80)


## Part E — What You Just Built

You built a small but real agentic system:

- **Tool calling:** the model selected a function and supplied structured arguments.
- **ReAct loop:** the model reasoned about needing data, acted through a tool, observed the result, and answered.
- **External API tool:** the model requested weather data through your code.
- **SQL agent:** natural-language questions became safe SQL queries over a database.

The model did not magically become connected to a database. You connected it by exposing one controlled tool. That is the deployment pattern.

## Production Guardrails

Before pointing this pattern at real data, add stronger controls:

- Use a read-only database user.
- Keep an allowlist of tables and columns.
- Parse SQL with a SQL parser rather than string checks.
- Add row limits and timeouts.
- Log every generated query.
- Add human review for sensitive domains.
- Trace tool calls with observability tools.

## Lab 1 Part 2 Complete

You should now have:

- [ ] A working one-tool memory sizing agent.
- [ ] A clear mental model for ReAct.
- [ ] A working external API tool call.
- [ ] A working SQL agent from scratch.
- [ ] A list of safety controls required for production SQL agents.

## Stretch Goals

1. Add a `cost_per_1k_tokens` column and ask for the cheapest high-quality model.
2. Add a second tool called `describe_schema()` and let the model inspect the schema before querying.
3. Add a stricter validator that rejects `SELECT *`.
4. Compare this scratch implementation with a LangChain SQL agent.
